# 5장 1강: A/B 테스트 설계 원리와 랜덤화 — 실습문제

## 실습 목표

- A/B 테스트의 대조군, 실험군, 랜덤화 단위를 데이터에서 식별합니다.
- 핵심 지표, 보조 지표, 가드레일 지표를 실험 목적에 맞게 사전에 정의합니다.
- 그룹 배정 수와 비율을 확인하고 계획한 50:50 배정과 일치하는지 점검합니다.
- 가설 → 설계 → 실행 → 분석의 순서로 온라인 실험계획을 작성합니다.

## 실습 환경 / 데이터

- Python, NumPy, pandas, SciPy
- `cookie_cats.csv`
- `userid`: 사용자 식별자
- `version`: 게임 게이트 위치(`gate_30`, `gate_40`)
- `sum_gamerounds`: 실험 기간 동안 플레이한 게임 라운드 수
- `retention_1`, `retention_7`: 설치 후 1일·7일 재방문 여부

> 이번 강의는 **실험 설계와 랜덤화 점검**이 중심입니다. 그룹 간 효과의 통계적 검정과 최종 배포 결정은 이후 강의에서 다룹니다.

## 실습 준비

아래 셀을 실행하여 데이터를 불러오고 크기, 결측치, 컬럼을 확인하세요.


In [5]:
# A/B 테스트 : 대상을 무작위로 나누어 서로 다른 두 버전을 제공하고 결과를 비교하는 실험
# -> A에는 회원가입, B에는 무료로 시작 버튼을 보여주고 같은 기간의 가입률을 비교

# 가드레일 지표 : 실험으로 인해 허용하기 어려운 부작용이 생기는지 확인하는 지표
# -> 추천영상 시청 완료율이 올라도 앱 이탈율이 크게 높아지는 현상이 발생한다면 부작용을 확인해야함

# 전환율과 클릭율 : 전환율 : 정한 목표 행동의 발생 비율 / 클릭율 : 정한 노출 대상중 클릭이 발생한 비율
# -> 대상 사용자 1,000명중 120명이 한 번 이상 구매했다면 사용자 기준 구매 전환율은 약 12%정도다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

data_candidates = [
    Path("cookie_cats.csv"),
    Path("upload/cookie_cats.csv")]

data_path = next((path for path in data_candidates if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("cookie_cats.csv 파일을 노트북과 같은 폴더에 넣어주세요.")

df = pd.read_csv(data_path)
alpha = 0.05

print(f"데이터 크기: {df.shape[0]}행, {df.shape[1]}열")
print("전체 결측치 수:", int(df.isna().sum().sum()))
print("컬럼:", df.columns.tolist())
df.head()


데이터 크기: 90189행, 5열
전체 결측치 수: 0
컬럼: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


---

## 필수 1. Cookie Cats 실험 구조와 지표 정의

게임의 첫 번째 강제 대기 게이트를 30단계에서 40단계로 옮기면 사용자 유지율이 달라지는지 확인하려고 합니다.

### 수행 요구사항

1. `userid`의 중복 여부를 확인하여 사용자 한 명이 한 행으로 기록되었는지 점검하세요.
2. `version`의 고유값과 그룹별 사용자 수를 확인하세요.
3. 버전별 사용자 수, 평균 게임 라운드 수, 1일 유지율, 7일 유지율을 하나의 요약표로 만드세요.
4. 아래 질문에 문장으로 답하세요.

### 질문

- 이 실험의 랜덤화 단위는 무엇인가요?
-> userid로 구분되는 사용자, 한 사용자는 하나의 버전에만 배정되어 일관된 게임 경험을 제공하여 비교

- `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
-> 기존 버전인 gate_30가 대조군 / 새로운 버전으로 옮긴 gate_40이 실험군으로 설정

- 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
-> 핵심 지표 : 단순 1일 평가보다는 7일 이후 시점을 나타내는 retention_7로(재방문)
-> 보조 지표 : 초기 반응 확인을 위해 retention_1로
-> 가드레일 지표 : 활동이 감소하는지 확인하기 위해 sum_gamerounds(평균 게임 라운드 수)로 보겠다.

- 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?
-> 아닙니다. 표본 변동으로 생길 수 있는 차이인지? 통계적으로 검정이 필요함

In [6]:
# 여기에 코드를 작성하세요.
# 1. userid 중복 여부 확인
n_total = len(df)
n_unique = df["userid"].nunique()
print(f"전체 행 수: {n_total}, 고유 userid 수: {n_unique}")
print(f"중복 여부: {'중복 있음' if n_total != n_unique else '중복 없음 (한 명당 한 행)'}")

# 2. version 고유값 및 그룹별 사용자 수
print(df["version"].unique())
print(df["version"].value_counts())

# 3. 버전별 요약표
summary = df.groupby("version").agg(
    사용자수=("userid", "count"),
    평균게임라운드=("sum_gamerounds", "mean"),
    유지율_1일=("retention_1", "mean"),
    유지율_7일=("retention_7", "mean"),
).round(4)
display(summary)

전체 행 수: 90189, 고유 userid 수: 90189
중복 여부: 중복 없음 (한 명당 한 행)
<StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: str
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64


,사용자수,평균게임라운드,유지율_1일,유지율_7일
version,,,,
gate_30,44700,52.4563,0.4482,0.1902
gate_40,45489,51.2988,0.4423,0.1820


---

## 필수 2. 무작위 배정 실습과 그룹 비율 점검

실제 `version` 값은 변경하지 않고, 동일한 사용자 목록에 연습용 A/B 그룹을 새로 무작위 배정한 뒤 실제 배정 비율과 비교하세요.

### 수행 요구사항

1. `np.random.default_rng(42)`를 사용하세요.
2. 각 `userid`에 `A` 또는 `B`를 50:50 확률로 배정한 `randomized_users`를 만드세요.
3. 연습용 그룹별 인원수와 비율을 출력하세요.
4. 실제 `version`별 인원수와 비율도 출력하세요.
5. 실제 실험이 50:50 배정을 계획했다고 가정하고, 기대빈도를 전체 인원의 절반으로 설정하여 카이제곱 적합도 검정을 수행하세요.
6. 아래 질문에 답하세요.

### 질문

- 시드를 고정하는 이유는 무엇인가요?
-> 무작위 과정을 재현하여 수강생과 검토자가 같은 배정 결과를 얻고 코드를 확인할 수 있게 하기 위해서

- 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
-> userid 별로 practice_group의 고유값 수를 계산하여 최대값이 1인지 확인

- 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
-> p-value가 0.05보다 작으므로, 50:50 계획과 통계적으로 일치한다고 보기 어렵다.

- `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?
-> 세 변수는 버전을 경험한 이후릐 측정된 결과이므로 처치영향을 받을 수 있다.
-> 균형점검에서는 실험 전에 측정된 기기, 국가 등 외부 요인같은 공변량이 필요하지만 현 데이터에서는 제공하지 않음

In [9]:
# 여기에 코드를 작성하세요.

# 1. rng 생성
rng = np.random.default_rng(42)

# 2. 연습용 A/B 그룹 50:50 무작위 배정
randomized_users = pd.DataFrame({
    "userid": df["userid"],
    "practice_group": rng.choice(["A", "B"], size=len(df), p=[0.5, 0.5]),
})

# 3. 연습용 그룹별 인원수, 비율
practice_counts = randomized_users["practice_group"].value_counts()
practice_ratio = randomized_users["practice_group"].value_counts(normalize=True)
print(practice_counts)
print(practice_ratio.round(4))

# 4. 실제 version별 인원수, 비율
version_counts = df["version"].value_counts()
version_ratio = df["version"].value_counts(normalize=True)
print(version_counts)
print(version_ratio.round(4))

# 5. 카이제곱 적합도 검정 (실제 version이 50:50 계획과 맞는지)
observed = version_counts.values
expected = np.array([len(df) / 2, len(df) / 2])
chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)
print(f"chi2 = {chi2_stat:.4f}, p-value = {p_value:.4e}")

alpha = 0.05
print(f"유의수준 0.05에서 유의함(50:50과 다름): {p_value < alpha}")

practice_group
B    45337
A    44852
Name: count, dtype: int64
practice_group
B    0.5027
A    0.4973
Name: proportion, dtype: float64
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64
version
gate_40    0.5044
gate_30    0.4956
Name: proportion, dtype: float64
chi2 = 6.9024, p-value = 8.6080e-03
유의수준 0.05에서 유의함(50:50과 다름): True


---

## 과제. Cookie Cats A/B 테스트 전체 계획서 작성

`gate_30`을 기존 버전, `gate_40`을 새 버전으로 설정한 A/B 테스트 계획을 작성하세요.

### 수행 요구사항

1. 아래 네 단계를 모두 포함한 계획서를 작성하세요.
   - 가설 설정
   - 실험 설계
   - 실험 실행
   - 결과 분석
2. 실험 단위, 대조군, 실험군, 핵심·보조·가드레일 지표를 명시하세요.
3. 실행 전에 확인할 데이터 품질 항목을 두 가지 이상 작성하세요.
4. SUTVA 위반 또는 실험 간 간섭 가능성을 검토하세요.
5. 버전별 관측 지표를 다시 계산하되, 아직 통계적 검정을 하지 않았다는 점을 반영해 최종 의사결정을 보류하는 6~8문장의 결론을 작성하세요.

> 과제는 필수 문제와 동일한 수준입니다. 표본 크기나 MDE를 계산할 필요는 없습니다.


In [10]:
# 여기에 코드를 작성하세요.
# 1-1. [가설 설정]
# 귀무가설(H0) : 게임 버전을 gate_30에서 gate_40으로 옮겨도 사용자 7일 유지율에는 차이가 없다.
# 대립가설(H1) : 게임 버전을 gate_30에서 gate_40으로 옮기면 사용자 7일 유지율에는 차이가 있다.

# 1-2. [실험 설계] 2번
# 실험 단위 : 랜덤시드
# 대조군 : gate_30 (기존 버전)
# 실험군 : gate_40 (새로운 버전으로 옮김)
# 핵심 지표 : retention_7 (단순 1일 평가보다는 7일 이후 시점을 나타내는 재방문으로)
# 보조 지표 : retention_1 (초기 반응 확인을 위해)
# 가드레일 지표 : sum_gamerounds (활동이 감소하는지 확인하기 위해(평균 게임 라운드 수)로 봄)

# 1-3. [실험 실행] 3번, 4번
# 3-1. 50:50 적합도와 실제 관측된 gate30/40 비율이 유의하게 다르지 않은지 확인
# 3-2. 중복이나 결측치가 있는지 확인
# 4. 같은 사람이 여러 아이디를 쓰는 경우 각각 다른 게이트를 쓰게될 가능성이 있다.

# 1-4 [결과분석]
# gate_30의 1일 유지율과 7일 유지율이 > gate_40이며, 평균 게임 라운드 수는 두 그룹이 비슷한 경향이 있다.
# 하지만 차이가 보인다는 사실만으로 `gate_40`의 차이의 효과라고 결론 내릴 수 없으며, 
# 표본 변동만으로도 우연히 생길 수 있기 때문에 통계적으로 검정이 필요하다.
# 검정에 들어가기 앞서, 실제 배정 비율이 50:50과 맞는지 카이제곱 검정으로 확인해보니 p-vlaue값이 0.05보다 작은 것으로 나타났다.
# 즉 실제 배정 비율은 50:50과 통계적으로 일치한다고 보기 어려우며, 검정을 진행하기 전에 이 배정 편향의 원인을 함께 점검할 필요가 있다.
# 결론적으로는 현재 시점에서는 게임 버전을 gate_30에서 gate_40으로 옮겨도 사용자 7일 유지율에는 차이가 있는지 말할 근거가 부족하며
# 따라서 편향의 원인과 통계적 검정을 하기 전까지는 최종 의사 결정을 보류한다.



---

## 실습 마무리

- 어떤 문제가 있었는가?
-> 그룹별 유지율과 평균 게임 라운드만 비교하여 사용자의 고정 배정 여부, 계획한 표본 비육, 지표사전 지정
-> 외부 간섭, 로깅 등의 문제를 놓칠 수 있었다.

- 어떻게 개선했는가?
-> 실험 단위를 userid로 명시하고 대조군, 실험군과 3가지 지표를 사전에 정의함
-> 실제 배정 비율, 이용자 중복 체크, 50:50 적합도, 결과변수, 사전 공변량의 차이 등을 함께 점검함

- 무엇을 근거로 개선되었다고 판단했는가?
-> 사용자 중복 0건, 실제 그룹비율 약 49.5%, 50.4%, 적합도 검정 p-value가 0.05 미만을 확인하여 단순히 비슷해보인다는 필수 1의 판단보다는 구체적인 조사 증거를 확보했다.
-> 40gate를 배포 전에 기술검정을 통해 버전 업 배포 결정을 보류했다.

단순한 그룹별 지표 비교에서 놓칠 수 있는 문제와, 실험 단위·사전 지표·배정 비율·간섭 가능성을 명시하면서 설계가 어떻게 개선되었는지 정리하세요.
